In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
import numpy as np

# --- CONFIGURATION ---
INPUT_FILE = 'READY_FOR_DL_MASTER.csv'
OUTPUT_FILE = 'STEP1_IMPUTED.csv'

# 1. LOAD DATA
print(f"Loading data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE, index_col='Date')
df = df.apply(pd.to_numeric, errors='coerce') # Ensure all columns are numeric

# 2. IMPUTATION (Median Fill)
# Robust step, even with 0 missing values, for pipeline safety.
print("Applying SimpleImputer (median strategy)...")
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(df)

# Convert back to DataFrame
df_imputed = pd.DataFrame(
    X_imputed,
    columns=df.columns,
    index=df.index
)

# 3. SAVE INTERMEDIATE FILE
df_imputed.to_csv(OUTPUT_FILE)
print(f"\nSUCCESS! Imputation complete. Data shape: {df_imputed.shape}")
print(f"Saved imputed features to: {OUTPUT_FILE}")

Loading data from READY_FOR_DL_MASTER.csv...
Applying SimpleImputer (median strategy)...

SUCCESS! Imputation complete. Data shape: (8122, 21)
Saved imputed features to: STEP1_IMPUTED.csv


In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

# --- CONFIGURATION ---
INPUT_FILE = 'STEP1_IMPUTED.csv'
OUTPUT_FILE = 'STEP2_SCALED.csv'

# 1. LOAD DATA
print(f"Loading imputed data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE, index_col='Date')

# 2. SCALING (StandardScaler)
# Critical step for neural networks: centering data around mean=0, std=1.
print("Applying StandardScaler (Z-Score normalization)...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Convert back to DataFrame
df_scaled = pd.DataFrame(
    X_scaled,
    columns=df.columns,
    index=df.index
)

# 3. SAVE INTERMEDIATE FILE
df_scaled.to_csv(OUTPUT_FILE)
print(f"\nSUCCESS! Scaling complete. Data shape: {df_scaled.shape}")
print(f"Saved scaled features to: {OUTPUT_FILE}")

Loading imputed data from STEP1_IMPUTED.csv...
Applying StandardScaler (Z-Score normalization)...

SUCCESS! Scaling complete. Data shape: (8122, 21)
Saved scaled features to: STEP2_SCALED.csv


In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm # Library for progress bar (optional, but helpful)

# --- CONFIGURATION ---
INPUT_FILE = 'STEP2_SCALED.csv'
OUTPUT_FILE_X = 'STEP3_X_3D.npy'
LOOKBACK_STEPS = 60  # The length of the historical window for the model

# --- FUNCTION FOR WINDOWING ---
def create_sequences(data, lookback):
    X = []
    # Iterate until the end of the available data minus the lookback
    for i in tqdm(range(len(data) - lookback + 1), desc="Creating windows"):
        # Input (X) is the sequence of 'lookback' steps
        X.append(data.iloc[i : (i + lookback)].values)
    return np.array(X)

# 1. LOAD DATA
print(f"Loading scaled data from {INPUT_FILE}...")
df_scaled = pd.read_csv(INPUT_FILE, index_col='Date')

# 2. WINDOWING
print(f"Applying Windowing (Lookback={LOOKBACK_STEPS})...")
X_3D = create_sequences(df_scaled, LOOKBACK_STEPS)

# 3. SAVE FINAL TENSOR
np.save(OUTPUT_FILE_X, X_3D)

print(f"\nSUCCESS! Windowing complete.")
print(f"Final Input Tensor Shape: {X_3D.shape}")
print(f"Saved final 3D NumPy tensor to: {OUTPUT_FILE_X}")

Loading scaled data from STEP2_SCALED.csv...
Applying Windowing (Lookback=60)...


Creating windows: 100%|██████████| 8063/8063 [00:01<00:00, 7700.69it/s]



SUCCESS! Windowing complete.
Final Input Tensor Shape: (8063, 60, 21)
Saved final 3D NumPy tensor to: STEP3_X_3D.npy


In [4]:
import numpy as np

# --- CONFIGURATION ---
INPUT_FILE_X = 'STEP3_X_3D.npy'

# 1. LOAD FINAL TENSOR
print(f"Loading final input tensor from {INPUT_FILE_X}...")
X_3D = np.load(INPUT_FILE_X)

# 2. CONFIRM SHAPE
N_SAMPLES, N_STEPS, N_FEATURES = X_3D.shape

print("\n---------------------------------------------------")
print("✅ FINAL INPUT TENSOR CONFIRMATION")
print("---------------------------------------------------")
print(f"Total Sequences (Batch Size): {N_SAMPLES}")
print(f"Lookback Steps (Window Size): {N_STEPS}")
print(f"Number of PCA Features: {N_FEATURES}")
print(f"Final Tensor Shape: ({N_SAMPLES}, {N_STEPS}, {N_FEATURES})")
print("\nThis tensor is perfectly formatted for an LSTM Autoencoder, VAE, or Transformer model.")
print("Use: model.fit(X_3D, X_3D, ...)")
print("---------------------------------------------------")

Loading final input tensor from STEP3_X_3D.npy...

---------------------------------------------------
✅ FINAL INPUT TENSOR CONFIRMATION
---------------------------------------------------
Total Sequences (Batch Size): 8063
Lookback Steps (Window Size): 60
Number of PCA Features: 21
Final Tensor Shape: (8063, 60, 21)

This tensor is perfectly formatted for an LSTM Autoencoder, VAE, or Transformer model.
Use: model.fit(X_3D, X_3D, ...)
---------------------------------------------------


In [ ]:
import numpy as np
import pandas as pd

# --- CONFIGURATION ---
INPUT_FILE_X = 'STEP3_X_3D.npy'
OUTPUT_FILE = 'STEP4_ML_INPUT_2D.csv'

# 1. LOAD FINAL TENSOR
print(f"Loading 3D input tensor from {INPUT_FILE_X}...")
X_3D = np.load(INPUT_FILE_X)

# 2. FEATURE EXTRACTION (Conversion to 2D)
# Calculate the Mean across the Lookback Steps (axis=1) for each feature.
# Shape transforms from (N_Samples, 60, N_Features) to (N_Samples, N_Features)
print("Converting 3D tensor to 2D by calculating the mean across the 60-day window...")
X_2D_ML = np.mean(X_3D, axis=1)

# 3. CREATE COLUMN NAMES
N_FEATURES = X_2D_ML.shape[1]
# We assume the columns were ordered by regime group (EMP, INF, GRW, etc.)
# Since the exact original column names are not tracked in the numpy array, 
# we use generic feature labels.
column_names = [f'Mean_PCA_Feature_{i+1}' for i in range(N_FEATURES)]

# 4. SAVE FINAL ML INPUT
# We cannot recover the original dates without the index from the windowing script, 
# so we save it as a clean data matrix suitable for ML model training.
df_2d_ml = pd.DataFrame(X_2D_ML, columns=column_names)
df_2d_ml.to_csv(OUTPUT_FILE, index=False)

print(f"\nSUCCESS! ML 2D Input created.")
print(f"New ML Data Shape: {df_2d_ml.shape}")
print(f"Saved 2D ML Input (ready for KMeans/DBSCAN/RandomForest) to: {OUTPUT_FILE}")